![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 14 -- Challenge Lab: AutoEncoder on FashionMNIST

This challenge brings together everything you have learned so far into one end-to-end project.

You will:
1. Load and explore the FashionMNIST dataset
2. Build an AutoEncoder in PyTorch
3. Train it to reconstruct images
4. Visualize the latent space using dimensionality reduction
5. **Bonus:** Generate brand-new images from the decoder

| Concept | Where you learned it |
|---|---|
| Data loading, transforms, DataLoaders | Days 6, 9 |
| `nn.Module`, layers, activations | Days 6--8 |
| Training loop, loss, optimizer | Days 7--8 |
| MSE loss for reconstruction | Day 3 |
| Conv layers, pooling | Day 9 |
| PCA / t-SNE / UMAP | Day 14 |
| Matplotlib visualization | Every day |


---

### The Big Picture

An **AutoEncoder** has two halves:

```
Image (784 pixels)
       │
       ▼
  ┌─────────┐
  │ ENCODER │   Compresses the image into a small vector
  └────┬────┘
       │
       ▼
 Latent Code (e.g. 16 dimensions)
       │
       ▼
  ┌─────────┐
  │ DECODER │   Reconstructs the image from the small vector
  └────┬────┘
       │
       ▼
Reconstructed Image (784 pixels)
```

The network learns to squeeze all the important information about a T-shirt, sneaker, or bag into just 16 numbers -- and then rebuild the image from those 16 numbers alone.

---
## Setup

In [ ]:
!pip install umap-learn -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

plt.rcParams['figure.dpi'] = 120

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## Part 1: Load and Explore FashionMNIST

FashionMNIST has the same format as MNIST (28×28 grayscale) but contains clothing items instead of digits. Here are the 10 classes:

| Label | Class |
|---|---|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

### Task 1: Load the Data

Load FashionMNIST using `torchvision.datasets.FashionMNIST`. Apply `transforms.ToTensor()` so each image becomes a tensor with values in [0, 1].

Create a `DataLoader` for the **training set** with `batch_size=256` and `shuffle=True`, and one for the **test set** with `batch_size=256` and `shuffle=False`.

Print the total number of training and test images.

In [ ]:
# Your code here

### Task 2: Visualize Samples

Display a grid of **16 random images** from the training set (4 rows × 4 columns). Show each image in grayscale and put the class name as the title of each subplot.

```
┌──────────┬──────────┬──────────┬──────────┐
│ T-shirt  │ Sneaker  │   Bag    │  Dress   │
│  [img]   │  [img]   │  [img]   │  [img]   │
├──────────┼──────────┼──────────┼──────────┤
│  Coat    │ Pullover │ Trouser  │  Sandal  │
│  [img]   │  [img]   │  [img]   │  [img]   │
├──────────┼──────────┼──────────┼──────────┤
│   ...    │   ...    │   ...    │   ...    │
└──────────┴──────────┴──────────┴──────────┘
```

In [ ]:
# Your code here

---
## Part 2: Build the AutoEncoder

You will build a **fully-connected** (linear) AutoEncoder. The architecture:

```
ENCODER                              DECODER
─────────                            ─────────
Input:  784                          Input:  16
  │                                    │
  ▼                                    ▼
Linear(784 → 256)                    Linear(16 → 64)
  │ ReLU                               │ ReLU
  ▼                                    ▼
Linear(256 → 64)                     Linear(64 → 256)
  │ ReLU                               │ ReLU
  ▼                                    ▼
Linear(64 → 16)                      Linear(256 → 784)
                                       │ Sigmoid
  ▼                                    ▼
Latent Code (16)                     Output (784)
```

- The **encoder** compresses 784 pixels → 16 numbers
- The **decoder** expands 16 numbers → 784 pixels
- Use `ReLU` between hidden layers
- Use `Sigmoid` at the decoder output (so pixel values stay in [0, 1])

### Task 3: Define the AutoEncoder Class

Create a class `AutoEncoder(nn.Module)` with:
- `self.encoder` -- a `nn.Sequential` with the encoder layers
- `self.decoder` -- a `nn.Sequential` with the decoder layers
- `forward(self, x)` that:
  1. Flattens the input from `(batch, 1, 28, 28)` to `(batch, 784)`
  2. Passes through the encoder to get the latent code
  3. Passes the latent code through the decoder
  4. Reshapes the output back to `(batch, 1, 28, 28)`
  5. Returns **both** the reconstructed image and the latent code

After defining the class, create an instance, move it to `device`, and print the model to verify the architecture.

In [ ]:
# Your code here

### Task 4: Sanity Check

Pass a single batch through the model and verify:
- The reconstructed output has the same shape as the input: `(batch, 1, 28, 28)`
- The latent code has shape `(batch, 16)`

Print both shapes.

In [ ]:
# Your code here

---
## Part 3: Train the AutoEncoder

### Task 5: Set Up Training

Define:
- **Loss function:** `nn.MSELoss()` -- measures how different the reconstruction is from the original
- **Optimizer:** `optim.Adam` with `lr=1e-3`
- **Number of epochs:** `20`

In [ ]:
# Your code here

### Task 6: Write the Training Loop

For each epoch:
1. Loop over batches from the training DataLoader
2. Move images to `device`
3. Forward pass: get the reconstructed images (ignore the latent codes during training)
4. Compute the MSE loss between the original images and the reconstructions
5. Backward pass and optimizer step
6. Track the average loss per epoch

At the end of each epoch, also compute the **test loss** (no gradients needed).

Print the train and test loss for each epoch.

```
Epoch  1/20  |  Train Loss: 0.0543  |  Test Loss: 0.0387
Epoch  2/20  |  Train Loss: 0.0351  |  Test Loss: 0.0318
  ...         ...                      ...
Epoch 20/20  |  Train Loss: 0.0198  |  Test Loss: 0.0195
```

In [ ]:
# Your code here

### Task 7: Plot the Loss Curves

Plot the training loss and test loss on the same figure, one line each. Add a legend, axis labels, and a title.

Is the model overfitting? How can you tell?

In [ ]:
# Your code here

### Task 8: Visualize Reconstructions

Take 10 images from the test set. Show them in two rows:
- **Top row:** Original images
- **Bottom row:** Reconstructed images from the AutoEncoder

How blurry are the reconstructions? Can you still tell what each item is?

In [ ]:
# Your code here

---
## Part 4: Visualize the Latent Space

### Task 9: Extract Latent Codes

Pass the **entire test set** through the encoder (no gradients needed). Collect:
- `all_codes` -- a NumPy array of shape `(10000, 16)` containing the latent codes
- `all_labels` -- a NumPy array of shape `(10000,)` containing the class labels

Print the shapes to verify.

In [ ]:
# Your code here

### Task 10: Reduce to 2D and Plot

Choose **one** of the three methods you learned today:
- `PCA(n_components=2)`
- `TSNE(n_components=2, perplexity=30, random_state=42)`
- `umap.UMAP(n_components=2, random_state=42)`

Apply it to `all_codes` to get a 2D projection. Then create a scatter plot:
- Each point is one test image
- Color the points by their class label (use `c=all_labels` and `cmap='tab10'`)
- Add a colorbar or legend showing which color is which class
- Title it with the method you chose

Do the classes form separate clusters? Which classes overlap?

In [ ]:
# Your code here

### Task 11: Try All Three Methods (Optional)

If you have time, apply all three methods (PCA, t-SNE, UMAP) and plot them side by side in a 1×3 figure, just like you did in Lab 3. Which method separates the clothing classes best?

In [ ]:
# Your code here

---
## Part 5 -- BONUS: Generate New Images!

Your decoder learned how to turn a vector of 16 numbers into a clothing image. What happens if you feed it **random** vectors that it has never seen before?

```
Random vector          Decoder          New image!
[0.3, -1.2, ...]  ──►   ┌─────┐  ──►    ┌───────┐
    (16 numbers)        │  D  │         │ ????? │
                        └─────┘         └───────┘
```

The decoder should produce images that look like clothing items -- even though these exact latent codes never came from a real image.

### Bonus Task A: Generate From Random Codes

1. Sample 16 random latent codes -- each is a vector of 16 numbers drawn from a standard normal distribution
2. Pass them through the **decoder only**
3. Display the 16 generated images in a 4×4 grid

```
Random latent codes (16 × 16)
           │
           ▼
    ┌─────────────┐
    │   DECODER   │
    └──────┬──────┘
           │
           ▼
   16 generated images
┌──────┬──────┬──────┬──────┐
│ img1 │ img2 │ img3 │ img4 │
├──────┼──────┼──────┼──────┤
│ img5 │ img6 │ img7 │ img8 │
├──────┼──────┼──────┼──────┤
│ img9 │ ...  │ ...  │ ...  │
├──────┼──────┼──────┼──────┤
│ ...  │ ...  │ ...  │img16 │
└──────┴──────┴──────┴──────┘
```

Do the generated images look like real clothing? Are some better than others?

In [ ]:
# Your code here

### Bonus Task B: Interpolate Between Two Items

Pick two real images (e.g., a sneaker and an ankle boot). Get their latent codes from the encoder. Then create **8 steps** of linear interpolation between the two codes, decode each one, and display them in a row.

You should see one item smoothly morph into the other.

```
Sneaker ─── step 1 ── step 2 ── step 3 ── ... ── step 8 ─── Ankle boot

 [img]     [img]     [img]     [img]     [img]    [img]     [img]     [img]
  100%      85%       71%       57%       43%      28%       14%        0%
sneaker   ◄─────────── gradually morphing ──────────────►   boot
```

In [ ]:
# Your code here

---
## Wrap-Up

**Congratulations!** You just built an end-to-end deep learning pipeline:

- **Data** → loaded and explored FashionMNIST
- **Model** → built an AutoEncoder with encoder + decoder
- **Training** → wrote a training loop with MSE loss and Adam
- **Evaluation** → visualized reconstructions and loss curves
- **Analysis** → used dimensionality reduction to see the latent space
- **Generation** → created new images from random codes

This is the same pipeline used in real-world generative AI -- just at a smaller scale.